# Multi-Armed Bandit

**Last updated:** 2026-04-09 12:00 PT

**Track:** Learning | **Sub-Ability:** Reinforcement learning

Can the model learn optimal strategies from reward feedback?
Pre/post learning framework: observe reward histories, identify best actions.

**Difficulty grid:** num arms × reward clarity × 3 seeds = 27 items

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import re
import json
import time

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)

In [ ]:
def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()

def extract_short_answer(response: str) -> str:
    """Extract a short answer (last short line, stripped of formatting)."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if not lines: return ''
    for line in reversed(lines):
        clean = line.strip('"\'\u2018\u2019\u201c\u201d').rstrip('.!?')
        if len(clean) <= 30:
            return clean.lower().strip()
    return lines[-1].strip('"\'\u2018\u2019\u201c\u201d').rstrip('.!?').lower().strip()

def extract_category(response: str) -> str:
    """Extract a category letter (A-D) from model response."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    for line in reversed(lines):
        m = re.search(r'\b([A-D])\b', line)
        if m and len(line) < 30: return m.group(1)
    for line in reversed(lines):
        m = re.search(r'([A-D])', line)
        if m: return m.group(1)
    return ''

In [ ]:
# Constants used in results analysis
NUM_ARMS = [2, 3, 4]
CLARITIES = ['clear', 'moderate', 'ambiguous']
SEEDS = 3

In [ ]:
# === Load dataset from Kaggle ===
import os, glob
candidates = glob.glob('/kaggle/input/**/multi_armed_bandit_dataset.csv', recursive=True)
if candidates:
    csv_path = candidates[0]
else:
    csv_path = '/kaggle/input/multi_armed_bandit_dataset.csv'
print(f'Loading from: {csv_path}')
dataset = pd.read_csv(csv_path)
print(f'Dataset: {len(dataset)} items')

In [ ]:
PRE_P = """You are playing a multi-armed bandit game. Your goal is to maximize reward.

{material}

Based on this history, which arm should you choose next to maximize reward?

Reply with ONLY the letter (e.g. A, B, C, or D). Nothing else."""

STUDY_P = """You are playing a multi-armed bandit game. Your goal is to maximize reward.

{material}

Analyze this data carefully:
1. Calculate the average reward (win rate) for each arm.
2. Identify which arm has the highest win rate.
3. Note any patterns or changes over time in the reward history.
4. Determine the best strategy going forward.

Write a clear analysis with your calculations."""

POST_P = """You previously analyzed a multi-armed bandit game and wrote these notes:

--- YOUR NOTES ---
{notes}
--- END NOTES ---

Here is the reward history again:
{material}

Based on your analysis and this history, which arm should you choose next to maximize reward?

Reply with ONLY the letter (e.g. A, B, C, or D). Nothing else."""

## Evaluation

In [ ]:
available = list(kbench.llms.keys())
all_results = []
@kbench.task(name='multi_armed_bandit',
             description='Pre/post learning multi-armed bandit — identify best arm from reward history')
def multi_armed_bandit(llm, material: str, test_input: str, expected: str,
                       num_arms: int, clarity: str, difficulty_label: str,
                       seed: int, item_desc: str, true_probs: str,
                       empirical_rates: str, num_plays: int, arms: str) -> dict:
    model_name = str(llm)
    ts = time.strftime('%Y-%m-%d %H:%M:%S')
    tid = f'{num_arms}arms_{clarity}_{seed}'

    t0 = time.time()
    pre_raw = llm.prompt(PRE_P.format(material=material))
    pre_time = time.time() - t0
    pre_extracted = extract_category(pre_raw)
    pre_correct = pre_extracted == expected

    t0 = time.time()
    study_raw = llm.prompt(STUDY_P.format(material=material))
    study_time = time.time() - t0
    notes = strip_thinking(study_raw)

    t0 = time.time()
    post_raw = llm.prompt(POST_P.format(notes=notes, material=material))
    post_time = time.time() - t0
    post_extracted = extract_category(post_raw)
    post_correct = post_extracted == expected

    all_results.append({
        'timestamp': ts, 'model': model_name, 'task_id': tid,
        'num_arms': int(num_arms), 'clarity': clarity, 'difficulty_label': difficulty_label,
        'seed': int(seed), 'item_desc': item_desc,
        'test_input': test_input, 'expected': expected,
        'true_probs': true_probs, 'empirical_rates': empirical_rates,
        'num_plays': int(num_plays), 'arms': arms,
        'pre_raw': pre_raw, 'pre_extracted': pre_extracted, 'pre_correct': int(pre_correct),
        'study_notes': notes,
        'post_raw': post_raw, 'post_extracted': post_extracted, 'post_correct': int(post_correct),
        'pre_time_s': pre_time, 'study_time_s': study_time, 'post_time_s': post_time
    })

    b,l = 'Y' if pre_correct else 'N', 'Y' if post_correct else 'N'
    print(f'[{model_name:40s}] [{difficulty_label:16s}] expected="{expected}"  pre="{pre_extracted}"({b})  post="{post_extracted}"({l})')
    return {'pre_correct': pre_correct, 'post_correct': post_correct}

try:
    runs = multi_armed_bandit.evaluate(llm=[kbench.llm], evaluation_data=dataset.set_index('task_id'),
                                    n_jobs=1, timeout=3600, max_attempts=1)
    print(f'\nDone: {len(runs.as_dataframe())} items')
except Exception as e:
    print(f'SDK post-processing error (non-fatal): {e}')

## Results

In [ ]:
results = pd.DataFrame(all_results)
print(f'Total runs: {len(results)}\n')

# Per-model summary
print('SCALING CHECK — Pre-learning accuracy by model:')
print('=' * 70)
for model in sorted(results['model'].unique()):
    sub = results[results['model'] == model]
    pre = sub['pre_correct'].mean()
    post = sub['post_correct'].mean()
    gain = post - pre
    print(f'  {str(model):45s}  pre={pre:.1%}  post={post:.1%}  gain={gain:+.1%}  (n={len(sub)})')

# Per model x clarity
print()
print('By model x clarity (PRE):')
print('-' * 70)
for model in sorted(results['model'].unique()):
    row = f'  {str(model):45s}'
    for c in CLARITIES:
        sub = results[(results['model'] == model) & (results['clarity'] == c)]
        if len(sub) > 0:
            row += f'  {c}={sub["pre_correct"].mean():.0%}'
    print(row)

# Per model x num_arms
print()
print('By model x num_arms (PRE):')
print('-' * 70)
for model in sorted(results['model'].unique()):
    row = f'  {str(model):45s}'
    for na in NUM_ARMS:
        sub = results[(results['model'] == model) & (results['num_arms'] == na)]
        if len(sub) > 0:
            row += f'  {na}arms={sub["pre_correct"].mean():.0%}'
    print(row)

print()
print(f'Timing: pre={results["pre_time_s"].mean():.1f}s  study={results["study_time_s"].mean():.1f}s  post={results["post_time_s"].mean():.1f}s')

In [ ]:
print('STUDY NOTES INSPECTION')
print('=' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    print(f'\n--- [{row["difficulty_label"]}] seed={row["seed"]} ---')
    print(f'Item: {row["item_desc"]}')
    print(f'True probs: {row["true_probs"]}')
    print(f'Empirical rates: {row["empirical_rates"]}')
    print(f'Expected: "{row["expected"]}"')
    print(f'Pre: "{row["pre_extracted"]}" ({b})  Post: "{row["post_extracted"]}" ({l})')
    print(f'Notes (first 300 chars): {row["study_notes"][:300]}')
    print('...' if len(str(row['study_notes'])) > 300 else '')

In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

labels = sorted(results['difficulty_label'].unique())
pre_scores = [results[results['difficulty_label']==l]['pre_correct'].mean() for l in labels]
post_scores = [results[results['difficulty_label']==l]['post_correct'].mean() for l in labels]

x = range(len(labels))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
ax1.bar([i-width/2 for i in x], pre_scores, width, label='Pre-learning', color='#4C72B0')
ax1.bar([i+width/2 for i in x], post_scores, width, label='Post-learning', color='#55A868')
ax1.set_ylabel('Accuracy')
ax1.set_title('Multi-Armed Bandit: Pre vs Post-Learning')
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [p-b for b,p in zip(pre_scores, post_scores)]
colors = ['#55A868' if g>=0 else '#C44E52' for g in gains]
ax2.bar(range(len(labels)), gains, color=colors)
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty')
ax2.set_xticks(list(x))
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('multi_armed_bandit_results.png', dpi=150, bbox_inches='tight')
plt.show()